In [3]:
import requests
import pandas as pd
from dotenv import load_dotenv
import os


load_dotenv()

print("Fetching data from CoinMarketCap...")

url = 'https://pro-api.coinmarketcap.com/v1/cryptocurrency/listings/latest'
headers = {
    'Accepts': 'application/json',
    'X-CMC_PRO_API_KEY': os.getenv('CMC_API_KEY')
}
params = {
    'start': '1',
    'limit': '5000',
    'convert': 'USD'
}

response = requests.get(url, headers=headers, params=params)

if response.status_code == 200:
    print(f"API call successful! Status: {response.status_code}")
else:
    print(f"API call FAILED. Status: {response.status_code}")
    print(response.text)
    exit()

df = pd.json_normalize(response.json()['data'])
print(f"{len(df)} cryptocurrencies fetched.")

print("Transforming data...")

insert_df = df[[
    'id', 'name', 'symbol',
    'quote.USD.price',
    'quote.USD.market_cap',
    'quote.USD.volume_24h',
    'quote.USD.percent_change_24h',
    'last_updated'
]].copy()

# Rename columns for cleanliness
insert_df.columns = [
    'id', 'name', 'symbol',
    'price_usd',
    'market_cap_usd',
    'volume_24h_usd',
    'percent_change_24h',
    'last_updated'
]

# Cast correct data types
insert_df['id']                 = insert_df['id'].astype(int)
insert_df['name']               = insert_df['name'].astype(str)
insert_df['symbol']             = insert_df['symbol'].astype(str)
insert_df['price_usd']          = insert_df['price_usd'].astype(float).round(4)
insert_df['market_cap_usd']     = insert_df['market_cap_usd'].astype(float).round(2)
insert_df['volume_24h_usd']     = insert_df['volume_24h_usd'].astype(float).round(2)
insert_df['percent_change_24h'] = insert_df['percent_change_24h'].astype(float).round(2)
insert_df['last_updated']       = insert_df['last_updated'].astype(str)

# Add price trend column
insert_df['price_trend'] = insert_df['percent_change_24h'].apply(
    lambda x: 'gaining' if x > 0 else ('losing' if x < 0 else 'stable')
)

# Drop any rows with nulls
insert_df.dropna(inplace=True)

print(f"Transformation complete. {len(insert_df)} clean rows ready.")
print(insert_df.head())

# 4. LOAD Save as CSV 
print("Saving to CSV...")

csv_path = os.path.join(os.path.dirname(os.path.abspath("pipeline.ipynb")), "crypto_data.csv")
insert_df.to_csv(csv_path, index=False)

print(f"Done! CSV saved to: {csv_path}")
print(f"Total rows: {len(insert_df)}")
print("\nNext step: Upload crypto_data.csv to Databricks via Data Ingestion")

Fetching data from CoinMarketCap...
API call successful! Status: 200
5000 cryptocurrencies fetched.
Transforming data...
Transformation complete. 5000 clean rows ready.
     id         name symbol   price_usd  market_cap_usd  volume_24h_usd  \
0     1      Bitcoin    BTC  66314.5119    1.326055e+12    4.079179e+10   
1  1027     Ethereum    ETH   1976.8114    2.385858e+11    2.296873e+10   
2   825  Tether USDt   USDT      1.0002    1.836731e+11    8.374721e+10   
3  1839          BNB    BNB    620.9067    8.466605e+10    1.852981e+09   
4    52          XRP    XRP      1.3647    8.336788e+10    3.833797e+09   

   percent_change_24h              last_updated price_trend  
0                3.62  2026-03-01T13:39:00.000Z     gaining  
1                5.86  2026-03-01T13:39:00.000Z     gaining  
2                0.02  2026-03-01T13:39:00.000Z     gaining  
3                4.03  2026-03-01T13:39:00.000Z     gaining  
4                4.98  2026-03-01T13:39:00.000Z     gaining  
Saving t